# 04 — Feature Importance

**Phase 4 (in-depth analysis).** Rank churn drivers with two complementary
models: (a) L2-regularized logistic regression (coefficient magnitude on
standardized features) and (b) a random forest (Gini feature_importances_).

**Leakage/target exclusions:** `customer_id`, `customer_status`, `churn`,
`churn_category`, `churn_reason`. `total_revenue` dropped (r = 0.97 with
`total_charges`); `total_charges` kept. `internet_type` is represented by
the engineered `is_fiber` + `internet_service` flags (avoids perfect
collinearity in the logistic model).

Population = existing customers only (`customer_status != 'Joined'`);
target = `churn` (28.4% positive).


In [ ]:
import matplotlib
matplotlib.use('Agg')
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
REPO = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
CLEAN = os.path.join(REPO, 'data/processed/clean_customers.csv')
FIG = os.path.join(REPO, 'reports/figures')
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 110
df = pd.read_csv(CLEAN)
df['offer'] = df['offer'].fillna('None')
existing = df[df['customer_status'] != 'Joined'].copy()
print('existing rows:', len(existing), '| positive rate:', round(existing['churn'].mean(),4))


## 1. Feature matrix


In [ ]:
# ---- Build feature matrix (leakage-free) --------------------------------
# EXCLUDED (leakage/target): customer_id, customer_status, churn,
#   churn_category, churn_reason.  total_revenue dropped (r=0.97 w/ total_charges).
bool_cols = ['married','phone_service','multiple_lines','internet_service',
             'online_security','online_backup','device_protection_plan',
             'premium_tech_support','streaming_tv','streaming_movies',
             'streaming_music','unlimited_data','paperless_billing','is_fiber']
num_cols = ['age','num_dependents','num_referrals','tenure_months',
            'avg_monthly_long_distance_charges','avg_monthly_gb_download',
            'monthly_charge','total_charges','total_refunds',
            'total_extra_data_charges','total_long_distance_charges',
            'contract_commitment','bundle_count']
cat_cols = ['offer','payment_method','contract','gender']
X = pd.concat([
    existing[num_cols].reset_index(drop=True),
    existing[bool_cols].astype(int).reset_index(drop=True),
    pd.get_dummies(existing[cat_cols], prefix=cat_cols, drop_first=True, dtype=int).reset_index(drop=True),
], axis=1)
y = existing['churn'].astype(int).reset_index(drop=True)
print('feature matrix:', X.shape, '| n features:', X.shape[1])


## 2. Train / evaluate

Stratified 80/20 split (random_state=42). LogisticRegression
(`class_weight='balanced'`, `max_iter=1000`) on standardized features;
RandomForestClassifier (`n_estimators=300`, `class_weight='balanced'`).


In [ ]:
# ---- Train / evaluate (stratified 80/20) --------------------------------
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
lr = Pipeline([('scaler', StandardScaler()),
               ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])
lr.fit(Xtr, ytr)
lr_auc = roc_auc_score(yte, lr.predict_proba(Xte)[:,1])
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
rf_auc = roc_auc_score(yte, rf.predict_proba(Xte)[:,1])
print('LogisticRegression ROC-AUC: %.4f' % lr_auc)
print('RandomForest ROC-AUC: %.4f' % rf_auc)
coefs = pd.Series(lr.named_steps['clf'].coef_[0], index=X.columns)
imps = pd.Series(rf.feature_importances_, index=X.columns)


## 3. Importance rankings


In [ ]:
# ---- Feature importance -------------------------------------------------
lr_top = coefs.abs().sort_values(ascending=False)
rf_top = imps.sort_values(ascending=False)
print('Top 15 by |logistic coefficient|:')
print(lr_top.head(15).round(4).to_string())
print('Top 15 by RandomForest feature_importances_:')
print(rf_top.head(15).round(6).to_string())
combined = pd.DataFrame({'lr_abs_coef': coefs.abs(), 'rf_importance': imps})
combined['lr_rank'] = combined['lr_abs_coef'].rank(ascending=False)
combined['rf_rank'] = combined['rf_importance'].rank(ascending=False)
combined['avg_rank'] = (combined['lr_rank'] + combined['rf_rank']) / 2
combined = combined.sort_values('avg_rank')
print('Combined ranking (top 15):')
print(combined[['lr_abs_coef','rf_importance','lr_rank','rf_rank','avg_rank']].head(15).round(4).to_string())


## 4. Figure


In [ ]:
# ---- Figure: top-15 feature importance side-by-side ---------------------
os.makedirs(FIG, exist_ok=True)
fig, axes = plt.subplots(1,2,figsize=(15,8))
l15 = lr_top.head(15)[::-1]
axes[0].barh(l15.index, l15.values, color='#1f77b4')
axes[0].set_title('Logistic regression — top 15 |coefficient|')
axes[0].set_xlabel('|standardized coefficient|')
r15 = rf_top.head(15)[::-1]
axes[1].barh(r15.index, r15.values, color='#d62728')
axes[1].set_title('RandomForest — top 15 feature_importances_')
axes[1].set_xlabel('Gini importance')
fig.suptitle('Churn feature importance', y=1.0)
fig.tight_layout()
fig.savefig(os.path.join(FIG,'stage4_feature_importance.png'), bbox_inches='tight')
plt.close(fig)
